In [ ]:
!pip uninstall -y onnxruntime onnxruntime-gpu -q
!pip install insightface scikit-learn scikit-image opencv-python pandas matplotlib tqdm scipy -q
!pip uninstall -y onnxruntime -q

# Version PIN kiya — random latest nahi, isse CUDA 13 wala masla nahi hoga
!pip install onnxruntime-gpu==1.19.2 -q

# cu11 ki jagah cu12 packages (1.19.2 CUDA 12 target karti hai)
!pip install nvidia-cublas-cu12 nvidia-cudnn-cu12 nvidia-curand-cu12 nvidia-cufft-cu12 -q

In [ ]:
import os
site_packages = '/usr/local/lib/python3.12/dist-packages'
cuda_lib_paths = [f'{site_packages}/nvidia/{p}/lib' for p in ['cublas', 'cudnn', 'curand', 'cufft']]
os.environ['LD_LIBRARY_PATH'] = ':'.join(cuda_lib_paths) + ':' + os.environ.get('LD_LIBRARY_PATH', '')

import onnxruntime as ort
providers = ort.get_available_providers()
print(providers)
assert 'CUDAExecutionProvider' in providers, "Still failing — paste full output back."
print("GPU confirmed available.")

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import cv2
import urllib.request
from tqdm import tqdm
from sklearn.datasets import fetch_lfw_pairs
from sklearn.metrics import roc_curve, auc
from scipy import stats
import pandas as pd

from insightface.app import FaceAnalysis

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

face_app = FaceAnalysis(
    name='buffalo_l',
    allowed_modules=['detection', 'recognition'],  # skip landmark_3d_68, landmark_2d_106,
                                                      # genderage — we never use these, and
                                                      # loading/running them was wasting real
                                                      # time on every single image
    providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
face_app.prepare(ctx_id=0, det_size=(320, 320))  # LFW faces are already tightly aligned/centered,
                                                    # so 320 is plenty — 640 was oversized for this data

In [ ]:
import pandas as pd
import glob
from PIL import Image

PAIRS_CSV = '/kaggle/input/datasets/jessicali9530/lfw-dataset/pairs.csv'
IMAGE_ROOT = '/kaggle/input/datasets/jessicali9530/lfw-dataset'

df = pd.read_csv(PAIRS_CSV)
assert len(df) == 6000, f"Expected 6000 rows, got {len(df)}"

# Build a filename -> full path index ONCE (13000+ files), instead of glob-searching
# per lookup (that was doing a full recursive directory scan per image — 6000x
# slower than necessary, hence the 26s/it you saw).
print("Indexing image files (one-time scan)...")
all_jpgs = glob.glob(f"{IMAGE_ROOT}/**/*.jpg", recursive=True)
filename_to_path = {os.path.basename(p): p for p in all_jpgs}
print(f"Indexed {len(filename_to_path)} images.")

def find_image_path(person_name, img_num):
    filename = f"{person_name}_{int(img_num):04d}.jpg"
    return filename_to_path.get(filename)

def load_face_image(path, target_size=250):
    img = Image.open(path).convert('RGB')
    if img.size != (target_size, target_size):
        img = img.resize((target_size, target_size))
    return np.asarray(img, dtype=np.float32) / 255.0

pairs_list = []
labels_list = []
fold_idx_list = []
missing = 0

for i, row in tqdm(df.iterrows(), total=len(df), desc="Loading LFW images"):
    fold = i // 600
    is_matched = pd.isna(row['Unnamed: 3'])

    if is_matched:
        person1, img1 = row['name'], row['imagenum1']
        person2, img2 = row['name'], row['imagenum2']
        label = 1
    else:
        person1, img1 = row['name'], row['imagenum1']
        person2, img2 = row['imagenum2'], row['Unnamed: 3']
        label = 0

    path1 = find_image_path(person1, img1)
    path2 = find_image_path(person2, img2)

    if path1 is None or path2 is None:
        missing += 1
        pairs_list.append(None)
        labels_list.append(label)
        fold_idx_list.append(fold)
        continue

    img_a = load_face_image(path1)
    img_b = load_face_image(path2)
    pairs_list.append(np.stack([img_a, img_b]))
    labels_list.append(label)
    fold_idx_list.append(fold)

print(f"Missing/unfindable images: {missing} / {len(df)}")
assert missing < 50, f"STOP — {missing} images not found. Paste this output back."

valid_idx = [i for i, p in enumerate(pairs_list) if p is not None]
pairs = np.stack([pairs_list[i] for i in valid_idx])
labels = np.array([labels_list[i] for i in valid_idx])
fold_idx = np.array([fold_idx_list[i] for i in valid_idx])

print(f"Final: {len(pairs)} pairs loaded ({labels.sum()} matched, {(labels==0).sum()} mismatched)")
print(f"Pairs per fold: {np.bincount(fold_idx)}")

In [ ]:
print("dtype:", pairs.dtype)
print("min/max:", pairs.min(), pairs.max())

import matplotlib.pyplot as plt
plt.imshow(pairs[0][0])
plt.title("Pair 0, image 0 — should show a full face with margin")
plt.show()

test_img = (pairs[0][0] * 255).astype('uint8') if pairs.max() <= 1.0 else pairs[0][0].astype('uint8')
test_bgr = cv2.cvtColor(test_img, cv2.COLOR_RGB2BGR)
faces = face_app.get(test_bgr)
print(f"Faces detected on clean image: {len(faces)}")
assert len(faces) > 0, "STOP — detection still failing on clean image. Paste this output back before continuing."
print("Detection working. Safe to continue.")

In [ ]:
class AODNet(nn.Module):
    def __init__(self):
        super(AODNet, self).__init__()
        self.relu = nn.ReLU(inplace=True)
        self.e_conv1 = nn.Conv2d(3, 3, 1, 1, 0)
        self.e_conv2 = nn.Conv2d(3, 3, 3, 1, 1)
        self.e_conv3 = nn.Conv2d(6, 3, 5, 1, 2)
        self.e_conv4 = nn.Conv2d(6, 3, 7, 1, 3)
        self.e_conv5 = nn.Conv2d(12, 3, 3, 1, 1)

    def forward(self, x):
        x1 = self.relu(self.e_conv1(x))
        x2 = self.relu(self.e_conv2(x1))
        cat1 = torch.cat((x1, x2), 1)
        x3 = self.relu(self.e_conv3(cat1))
        cat2 = torch.cat((x2, x3), 1)
        x4 = self.relu(self.e_conv4(cat2))
        cat3 = torch.cat((x1, x2, x3, x4), 1)
        k = self.relu(self.e_conv5(cat3))
        output = k * x - k + 1
        output = self.relu(output)
        return output

print("AODNet defined")

In [ ]:
os.makedirs('/kaggle/working/weights', exist_ok=True)

# Pretrained
pretrained_path = '/kaggle/working/weights/aod_net.pth'
if not os.path.exists(pretrained_path):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/MayankSingal/PyTorch-Image-Dehazing/master/snapshots/dehazer.pth",
        pretrained_path)
aod_pretrained = AODNet().to(DEVICE)
aod_pretrained.load_state_dict(torch.load(pretrained_path, map_location=DEVICE))
aod_pretrained.eval()

# Fine-tuned (from your GitHub repo)
finetuned_path = '/kaggle/working/weights/aod_net_finetuned.pth'
if not os.path.exists(finetuned_path):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/mrsaqi321/RT-FAREP/main/models/aod_net_finetuned.pth",
        finetuned_path)
aod_finetuned = AODNet().to(DEVICE)
aod_finetuned.load_state_dict(torch.load(finetuned_path, map_location=DEVICE))
aod_finetuned.eval()

print("Both models loaded.")

In [ ]:
def apply_synthetic_fog(image, beta, atmospheric_light=0.9):
    t = np.exp(-beta)
    hazy = image * t + atmospheric_light * (1 - t)
    return np.clip(hazy, 0, 1).astype(np.float32)

def dehaze_batch(images, model, device=DEVICE):
    model.eval()
    with torch.no_grad():
        tensor = torch.from_numpy(images).permute(0, 3, 1, 2).float().to(device)
        output = model(tensor)
        dehazed = output.permute(0, 2, 3, 1).cpu().numpy()
    return np.clip(dehazed, 0, 1)

def extract_embedding(image_uint8):
    faces = face_app.get(image_uint8)
    if len(faces) == 0:
        return None
    largest = max(faces, key=lambda f: (f.bbox[2]-f.bbox[0]) * (f.bbox[3]-f.bbox[1]))
    emb = largest.embedding
    return emb / np.linalg.norm(emb)

def process_pairs_at_fog_level(pairs, beta, model, device=DEVICE, batch_size=128, apply_dehazing=True):
    n_pairs = len(pairs)
    embeddings_a = [None] * n_pairs
    embeddings_b = [None] * n_pairs
    failures = 0

    for i in tqdm(range(0, n_pairs, batch_size), desc=f"beta={beta}, dehaze={apply_dehazing}"):
        end = min(i + batch_size, n_pairs)
        batch_a = pairs[i:end, 0]
        batch_b = pairs[i:end, 1]

        hazy_a = np.stack([apply_synthetic_fog(img, beta) for img in batch_a])
        hazy_b = np.stack([apply_synthetic_fog(img, beta) for img in batch_b])

        if apply_dehazing:
            proc_a = dehaze_batch(hazy_a, model, device)
            proc_b = dehaze_batch(hazy_b, model, device)
        else:
            proc_a, proc_b = hazy_a, hazy_b

        for j in range(end - i):
            idx = i + j
            img_a_bgr = cv2.cvtColor((proc_a[j] * 255).astype(np.uint8), cv2.COLOR_RGB2BGR)
            img_b_bgr = cv2.cvtColor((proc_b[j] * 255).astype(np.uint8), cv2.COLOR_RGB2BGR)
            emb_a = extract_embedding(img_a_bgr)
            emb_b = extract_embedding(img_b_bgr)
            embeddings_a[idx] = emb_a
            embeddings_b[idx] = emb_b
            if emb_a is None or emb_b is None:
                failures += 1

    return {'embeddings_a': embeddings_a, 'embeddings_b': embeddings_b, 'detection_failures': failures}

print("Pipeline functions defined.")

In [ ]:
def compute_pair_similarities(emb_a, emb_b, labels):
    n = len(labels)
    sims = np.full(n, np.nan)
    valid = np.zeros(n, dtype=bool)
    for i in range(n):
        if emb_a[i] is not None and emb_b[i] is not None:
            sims[i] = float(np.dot(emb_a[i], emb_b[i]))
            valid[i] = True
    return sims[valid], labels[valid], valid, (n - valid.sum())

def tar_at_far(sims, labels, target_far=0.001):
    fpr, tpr, thresholds = roc_curve(labels, sims)
    idx_ok = np.where(fpr <= target_far)[0]
    if len(idx_ok) == 0:
        return tpr[0], thresholds[0], fpr[0]
    idx = idx_ok[-1]
    return tpr[idx], thresholds[idx], fpr[idx]

def evaluate_with_ci(sims, labels, fold_idx, target_far=0.001):
    n_folds = len(np.unique(fold_idx))
    tars, aucs = [], []
    for f in range(n_folds):
        mask = fold_idx == f
        tar, _, _ = tar_at_far(sims[mask], labels[mask], target_far)
        fpr, tpr, _ = roc_curve(labels[mask], sims[mask])
        tars.append(tar)
        aucs.append(auc(fpr, tpr))
    tars, aucs = np.array(tars), np.array(aucs)
    ci_tar = stats.t.interval(0.95, len(tars)-1, loc=tars.mean(), scale=stats.sem(tars))
    ci_auc = stats.t.interval(0.95, len(aucs)-1, loc=aucs.mean(), scale=stats.sem(aucs))
    return {'mean_tar': tars.mean(), 'ci_tar': ci_tar, 'mean_auc': aucs.mean(), 'ci_auc': ci_auc}

print("Benchmark functions defined.")

In [ ]:
# Run just ONE condition, ONE beta, to confirm everything works + estimate timing
import time
start = time.time()
test_result = process_pairs_at_fog_level(pairs, beta=2.5, model=None, apply_dehazing=False, batch_size=32)
elapsed = time.time() - start
print(f"Test run: {elapsed:.1f}s for 6000 pairs, raw condition, beta=2.5")
print(f"Detection failures: {test_result['detection_failures']} / 6000")
print(f"Estimated total time for all 18 passes: {elapsed*18/60:.1f} minutes")

In [ ]:
test_zero = process_pairs_at_fog_level(pairs, beta=0.01, model=None, apply_dehazing=False, batch_size=32)
print(f"Detection failures at near-zero fog: {test_zero['detection_failures']} / 6000")

In [ ]:
import pickle

BETA_LEVELS = [0.5, 1.5, 2.5, 3.0, 3.5, 4.0]
CHECKPOINT_PATH = '/kaggle/working/embeddings_checkpoint.pkl'

# Resume from checkpoint if one exists (protects against disconnects)
if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH, 'rb') as f:
        embeddings_by_condition = pickle.load(f)
    print(f"Resumed checkpoint with {len(embeddings_by_condition)} conditions already done: "
          f"{list(embeddings_by_condition.keys())}")
else:
    embeddings_by_condition = {}

for beta in BETA_LEVELS:
    # Skip any condition already saved in the checkpoint — this is what lets
    # you resume instead of restarting from beta=0.5 every time
    if f'raw_beta{beta}' in embeddings_by_condition:
        print(f"beta={beta} raw already done, skipping")
    else:
        r_raw = process_pairs_at_fog_level(pairs, beta, model=None, apply_dehazing=False)
        embeddings_by_condition[f'raw_beta{beta}'] = r_raw
        with open(CHECKPOINT_PATH, 'wb') as f:
            pickle.dump(embeddings_by_condition, f)

    if f'pretrained_beta{beta}' in embeddings_by_condition:
        print(f"beta={beta} pretrained already done, skipping")
    else:
        r_pre = process_pairs_at_fog_level(pairs, beta, model=aod_pretrained, apply_dehazing=True)
        embeddings_by_condition[f'pretrained_beta{beta}'] = r_pre
        with open(CHECKPOINT_PATH, 'wb') as f:
            pickle.dump(embeddings_by_condition, f)

    if f'finetuned_beta{beta}' in embeddings_by_condition:
        print(f"beta={beta} finetuned already done, skipping")
    else:
        r_fine = process_pairs_at_fog_level(pairs, beta, model=aod_finetuned, apply_dehazing=True)
        embeddings_by_condition[f'finetuned_beta{beta}'] = r_fine
        with open(CHECKPOINT_PATH, 'wb') as f:
            pickle.dump(embeddings_by_condition, f)

    print(f"beta={beta} fully done and checkpointed.")

print("ALL DONE.")

In [34]:
rows = []
for name, emb_data in embeddings_by_condition.items():
    sims, valid_labels, valid_mask, n_dropped = compute_pair_similarities(
        emb_data['embeddings_a'], emb_data['embeddings_b'], labels)
    valid_fold = fold_idx[valid_mask]
    result = evaluate_with_ci(sims, valid_labels, valid_fold)
    rows.append({
        'condition': name,
        'mean_tar': result['mean_tar'],
        'tar_ci_lower': result['ci_tar'][0],
        'tar_ci_upper': result['ci_tar'][1],
        'mean_auc': result['mean_auc'],
        'auc_ci_lower': result['ci_auc'][0],
        'auc_ci_upper': result['ci_auc'][1],
        'detection_failure_rate': n_dropped / len(labels),
    })

results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

results_df.to_csv('/kaggle/working/v2_tar_far_table.csv', index=False)
print("\nSaved to /kaggle/working/v2_tar_far_table.csv — download this and share it back.")

         condition  mean_tar  tar_ci_lower  tar_ci_upper  mean_auc  auc_ci_lower  auc_ci_upper  detection_failure_rate
       raw_beta0.5  0.976333      0.970444      0.982223  0.989556      0.986378      0.992733                0.000000
pretrained_beta0.5  0.978667      0.971365      0.985969  0.992429      0.989871      0.994987                0.000000
 finetuned_beta0.5  0.971048      0.962969      0.979127  0.991982      0.989125      0.994839                0.016667
       raw_beta1.5  0.978333      0.974399      0.982268  0.991104      0.988400      0.993809                0.000000
pretrained_beta1.5  0.979333      0.973966      0.984701  0.992622      0.990367      0.994878                0.000000
 finetuned_beta1.5  0.980667      0.974634      0.986699  0.992766      0.990472      0.995059                0.000000
       raw_beta2.5  0.968390      0.956912      0.979867  0.994667      0.990853      0.998480                0.053500
pretrained_beta2.5  0.982313      0.977535      

/usr/local/lib/python3.12/dist-packages/scipy/stats/_distn_infrastructure.py:2323: RuntimeWarning: invalid value encountered in multiply
  lower_bound = _a * scale + loc
/usr/local/lib/python3.12/dist-packages/scipy/stats/_distn_infrastructure.py:2324: RuntimeWarning: invalid value encountered in multiply
  upper_bound = _b * scale + loc
